# LeetCode 600 — Non-negative Integers without Consecutive Ones bằng Python

- Link: https://leetcode.com/problems/non-negative-integers-without-consecutive-ones/
- Độ khó: **Hard**
- Chủ đề: Thao tác bit, Quy hoạch động chữ số (Digit DP), Dãy Fibonacci, Đếm

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findIntegers(self, n: int) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài này là cửa ngõ vào **Digit DP** — kỹ thuật đếm số nguyên trong `[0, n]` thỏa một tính chất về chữ số (ở đây là **chữ số nhị phân**). Kết quả bất ngờ đẹp: số chuỗi bit độ dài `k` không có hai số 1 liền nhau chính là **số Fibonacci**.

Sau notebook này bạn sẽ:

1. Thành thạo vài thao tác bit cơ bản: `bin`, `>>`, `&`, `bit_length`, kiểm tra hai bit 1 liền nhau bằng `x & (x >> 1)`.
2. Chứng minh vì sao số chuỗi bit độ dài `k` "không có 11" là Fibonacci.
3. Hiểu kỹ thuật **"đi theo các bit của `n`, mỗi khi gặp bit 1 thì đếm cả nhánh đặt bit 0"** — trái tim của Digit DP.
4. Viết 4 lời giải: vét cạn, sinh trực tiếp các số hợp lệ, Digit DP có nhớ `(pos, prev, tight)`, và quét bit + Fibonacci `O(log n)`.
5. Kiểm chứng bằng brute force cho mọi `n` nhỏ và `n` ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho số nguyên dương `n`. Đếm số nguyên `x` trong đoạn `[0, n]` sao cho **biểu diễn nhị phân** của `x` **không có hai bit 1 đứng liền nhau**.

**Ví dụ 1**

```text
Input:  n = 5
Output: 5
Giải thích: các số 0..5 trong nhị phân:
0 : 0
1 : 1
2 : 10
3 : 11      <- có hai số 1 liền nhau, loại
4 : 100
5 : 101
Còn lại 5 số: 0, 1, 2, 4, 5.
```

**Ví dụ 2**

```text
Input:  n = 1
Output: 2
```

**Ví dụ 3**

```text
Input:  n = 2
Output: 3
```

**Ràng buộc**

- `1 <= n <= 10^9`

Với `n = 10^9`, duyệt từng số là 1 tỷ phép kiểm tra — quá chậm trong Python. Ta cần đếm thông minh.

In [1]:
for x in range(0, 9):
    s = bin(x)[2:]
    print(f"{x}: {s:>4}  {'có 11 -> loại' if '11' in s else 'hợp lệ'}")

0:    0  hợp lệ
1:    1  hợp lệ
2:   10  hợp lệ
3:   11  có 11 -> loại
4:  100  hợp lệ
5:  101  hợp lệ
6:  110  có 11 -> loại
7:  111  có 11 -> loại
8: 1000  hợp lệ


## 3. Kiến thức nền

### 3.1. `bin`, `bit_length` và đọc bit thứ `k`

- `bin(5)` = `'0b101'`; bỏ 2 ký tự đầu để được chuỗi bit.
- `n.bit_length()` = số bit cần để biểu diễn `n` (`5 → 3`).
- Bit thứ `k` (đếm từ 0 ở bên phải): `(n >> k) & 1`.

In [2]:
n = 5
print(bin(n), bin(n)[2:], n.bit_length())
print([(k, (n >> k) & 1) for k in range(n.bit_length() - 1, -1, -1)])

0b101 101 3
[(2, 1), (1, 0), (0, 1)]


### 3.2. Phát hiện hai bit 1 liền nhau: `x & (x >> 1)`

Dịch phải 1 bit làm mỗi bit "đứng cạnh" bit bên trái của nó. Nếu có cặp `11` ở đâu đó, phép AND sẽ có bit 1 tại đó → kết quả khác 0.

In [3]:
for x in [5, 6, 10, 11, 21]:
    print(f"x={x:>2} ({x:05b}) | x>>1 = {x >> 1:05b} | AND = {x & (x >> 1):05b} -> {'có 11' if x & (x >> 1) else 'không có 11'}")

x= 5 (00101) | x>>1 = 00010 | AND = 00000 -> không có 11
x= 6 (00110) | x>>1 = 00011 | AND = 00010 -> có 11
x=10 (01010) | x>>1 = 00101 | AND = 00000 -> không có 11
x=11 (01011) | x>>1 = 00101 | AND = 00001 -> có 11
x=21 (10101) | x>>1 = 01010 | AND = 00000 -> không có 11


### 3.3. Dãy Fibonacci

`F(0) = 1, F(1) = 2, F(k) = F(k-1) + F(k-2)` — ta sẽ dùng dãy "lệch" này (bắt đầu 1, 2) vì nó đếm đúng số chuỗi bit. Tạo bằng list và vòng lặp.

In [4]:
f = [1, 2]
for k in range(2, 12):
    f.append(f[-1] + f[-2])
print(f)

[1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233]


### 3.4. Đệ quy có nhớ với nhiều tham số (cho Digit DP)

`@lru_cache` lưu theo **bộ tham số** `(pos, prev, tight)`. Tham số kiểu `bool` cũng hash được.

In [5]:
from functools import lru_cache

@lru_cache(maxsize=None)
def count_paths(i, j, flag):
    if i == 0 and j == 0:
        return 1
    res = 0
    if i > 0: res += count_paths(i - 1, j, flag)
    if j > 0: res += count_paths(i, j - 1, not flag)
    return res

print(count_paths(5, 5, True), count_paths.cache_info().currsize)

252 36


### 3.5. Duyệt cây bằng stack (DFS lặp)

Sinh các số hợp lệ như duyệt một cây: mỗi số `x` có con `2x` (thêm bit 0) và `2x + 1` (thêm bit 1, chỉ khi bit cuối của `x` là 0). Dùng list làm stack để tránh đệ quy sâu.

In [6]:
stack, seen = [1], []
while stack:
    x = stack.pop()
    if x > 20:
        continue
    seen.append(x)
    stack.append(x << 1)
    if x & 1 == 0:
        stack.append((x << 1) | 1)
print(sorted(seen))

[1, 2, 4, 5, 8, 9, 10, 16, 17, 18, 20]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đếm chuỗi bit độ dài `k` không có `11` — tính tay

Gọi `F(k)` = số chuỗi bit độ dài `k` (cho phép số 0 ở đầu) không có hai số 1 liền nhau.

| k | các chuỗi hợp lệ | F(k) |
|---|---|---|
| 0 | (chuỗi rỗng) | 1 |
| 1 | `0`, `1` | 2 |
| 2 | `00`, `01`, `10` | 3 |
| 3 | `000`, `001`, `010`, `100`, `101` | 5 |
| 4 | `0000`, `0001`, `0010`, `0100`, `0101`, `1000`, `1001`, `1010` | 8 |

**Vì sao là Fibonacci?** Xét bit **đầu tiên** của chuỗi độ dài `k`:
- Là `0` → phần còn lại là chuỗi hợp lệ bất kỳ độ dài `k-1`: `F(k-1)` cách.
- Là `1` → bit thứ hai **bắt buộc** là `0`, phần còn lại độ dài `k-2` tùy ý: `F(k-2)` cách.

→ `F(k) = F(k-1) + F(k-2)`.

### 4.2. Đếm số `<= n`: tính tay `n = 10 = 1010₂`

Các số 4 bit `<= 1010` (cho phép số 0 đầu). Đi từ bit cao xuống:

- **Bit 3 của `n` là 1.** Nếu ta đặt bit 3 = **0**, thì 3 bit còn lại **tùy ý** (vì số đã chắc chắn nhỏ hơn `n`) → `F(3) = 5` số (`0000..0101` hợp lệ). Nếu đặt bit 3 = 1 (bám theo `n`), đi tiếp.
- **Bit 2 của `n` là 0.** Bắt buộc đặt 0, đi tiếp.
- **Bit 1 của `n` là 1.** Đặt 0 → bit 0 tùy ý: `F(1) = 2` số (`1000`, `1001`). Đặt 1 (bám `n`): đi tiếp. Prefix hiện tại `101` — vẫn hợp lệ.
- **Bit 0 của `n` là 0.** Bắt buộc 0.
- Hết bit: chính `n = 1010` hợp lệ → +1.

Tổng: `5 + 2 + 1 = 8`. Kiểm tra: `0, 1, 2, 4, 5, 8, 9, 10` ✓.

**Điểm dừng sớm:** nếu `n` có hai bit 1 liền nhau, khi bám theo `n` tới bit 1 thứ hai, ta đã đếm nhánh "đặt 0" rồi; nhánh "đặt 1" sẽ tạo `11` → **dừng ngay**, không cộng 1 cho `n` ở cuối.

In [7]:
def F_list(k):
    f = [1, 2]
    while len(f) <= k:
        f.append(f[-1] + f[-2])
    return f

f = F_list(10)
for k in range(6):
    brute = sum(1 for x in range(2 ** k) if x & (x >> 1) == 0)
    print(f"F({k}) = {f[k]}, đếm vét cạn = {brute}")

F(0) = 1, đếm vét cạn = 1
F(1) = 2, đếm vét cạn = 2
F(2) = 3, đếm vét cạn = 3
F(3) = 5, đếm vét cạn = 5
F(4) = 8, đếm vét cạn = 8
F(5) = 13, đếm vét cạn = 13


### 4.3. Bảng đếm dồn — quan sát "bậc thang Fibonacci"

In `n` từ 0 đến 20 cùng số lượng hợp lệ trong `[0, n]`. Để ý tại các lũy thừa của 2 (`n = 2^k - 1`, tức ngay trước khi cần thêm bit), đáp án đúng bằng `F(k)`: `1→2, 3→3, 7→5, 15→8` — đúng như mục 4.1, vì `[0, 2^k - 1]` là **mọi** chuỗi `k` bit.

In [8]:
f = F_list(10)
running = 0
for x in range(0, 21):
    ok = x & (x >> 1) == 0
    running += ok
    mark = f"  <- n = 2^{x.bit_length()} - 1, F({x.bit_length()}) = {f[x.bit_length()]}" if (x + 1) & x == 0 and x > 0 else ""
    print(f"x={x:>2} {x:>6b} {'hợp lệ' if ok else 'loại  '} | đếm trong [0, x] = {running}{mark}")
for k in range(1, 10):
    assert sum(1 for x in range(2 ** k) if x & (x >> 1) == 0) == f[k]

x= 0      0 hợp lệ | đếm trong [0, x] = 1
x= 1      1 hợp lệ | đếm trong [0, x] = 2  <- n = 2^1 - 1, F(1) = 2
x= 2     10 hợp lệ | đếm trong [0, x] = 3
x= 3     11 loại   | đếm trong [0, x] = 3  <- n = 2^2 - 1, F(2) = 3
x= 4    100 hợp lệ | đếm trong [0, x] = 4
x= 5    101 hợp lệ | đếm trong [0, x] = 5
x= 6    110 loại   | đếm trong [0, x] = 5
x= 7    111 loại   | đếm trong [0, x] = 5  <- n = 2^3 - 1, F(3) = 5
x= 8   1000 hợp lệ | đếm trong [0, x] = 6
x= 9   1001 hợp lệ | đếm trong [0, x] = 7
x=10   1010 hợp lệ | đếm trong [0, x] = 8
x=11   1011 loại   | đếm trong [0, x] = 8
x=12   1100 loại   | đếm trong [0, x] = 8
x=13   1101 loại   | đếm trong [0, x] = 8
x=14   1110 loại   | đếm trong [0, x] = 8
x=15   1111 loại   | đếm trong [0, x] = 8  <- n = 2^4 - 1, F(4) = 8
x=16  10000 hợp lệ | đếm trong [0, x] = 9
x=17  10001 hợp lệ | đếm trong [0, x] = 10
x=18  10010 hợp lệ | đếm trong [0, x] = 11
x=19  10011 loại   | đếm trong [0, x] = 11
x=20  10100 hợp lệ | đếm trong [0, x] = 12


### 4.4. Phân nhóm thực tế cho `n = 10`

Ta liệt kê các số hợp lệ `<= 10` và gắn mỗi số với **nhóm** = vị trí bit cao nhất mà nó khác `n` (bit đó của `n` là 1, của số là 0). Số lượng mỗi nhóm đúng bằng `F(k)`:

In [9]:
n = 10
groups = {}
for x in range(n + 1):
    if x & (x >> 1):
        continue
    if x == n:
        groups.setdefault("chính n", []).append(x)
        continue
    k = (x ^ n).bit_length() - 1           # bit cao nhất khác nhau
    groups.setdefault(f"khác ở bit {k}", []).append(x)
for g, xs in groups.items():
    print(f"{g:<14}: {[format(x, '04b') for x in xs]} -> {len(xs)} số")
assert len(groups["khác ở bit 3"]) == F_list(3)[3] and len(groups["khác ở bit 1"]) == F_list(1)[1]

khác ở bit 3  : ['0000', '0001', '0010', '0100', '0101'] -> 5 số
khác ở bit 1  : ['1000', '1001'] -> 2 số
chính n       : ['1010'] -> 1 số


## 5. Cách 1 — Brute Force: kiểm tra từng số

**Ý tưởng.** Duyệt `x = 0..n`, đếm các `x` có `x & (x >> 1) == 0`.

In [10]:
def find_integers_brute(n: int) -> int:
    return sum(1 for x in range(n + 1) if x & (x >> 1) == 0)

print([find_integers_brute(n) for n in range(1, 11)])
assert find_integers_brute(5) == 5
assert find_integers_brute(1) == 2
assert find_integers_brute(2) == 3

[2, 3, 3, 4, 5, 5, 5, 6, 7, 8]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Với `n = 10^9`: khoảng vài trăm giây trong Python → TLE.

**Nút thắt:** số lượng số **hợp lệ** ít hơn `n` rất nhiều (chỉ cỡ `F(30) ≈ 2.2 triệu` với `n = 10^9`), và hơn nữa ta chỉ cần **đếm**, không cần liệt kê.

In [11]:
import time
for n in (10**5, 10**6):
    t = time.perf_counter()
    r = find_integers_brute(n)
    print(f"n={n:>8}: {r:>6} số, {time.perf_counter() - t:.3f}s")

n=  100000:   4181 số, 0.004s
n= 1000000:  17711 số, 0.038s


## 6. Bước đột phá

**Insight 1 — Fibonacci đếm khối tự do.** Khi đã chắc chắn số đang xét **nhỏ hơn `n`** ở một bit cao nào đó, mọi bit thấp hơn là tự do → số cách = `F(số bit còn lại)`. Không cần duyệt.

**Insight 2 — Chỉ có `O(log n)` "điểm rẽ nhánh".** Mọi số `x < n` có một bit cao nhất mà tại đó `x` có 0 còn `n` có 1 (phía trên bit đó `x` giống hệt `n`). Nhóm các `x` theo bit đó: mỗi bit 1 của `n` là một nhóm, kích thước nhóm = `F(k)` nếu phần đầu (giống `n`) chưa có `11`.

**Quy tắc (quét bit từ cao xuống, `prev` = bit trước của `n`):**
- Nếu bit `k` của `n` là 1 thì `ans += F(k)` (nhóm "đặt 0 tại `k`").
  - Nếu `prev` cũng là 1 thì **dừng** và trả `ans` (bám theo `n` sẽ tạo `11`).
- Nếu quét hết mà không dừng thì `n` hợp lệ → `ans + 1`.

**Invariant:** trước khi xét bit `k`, phần đầu của `n` (các bit cao hơn `k`) **không chứa `11`**, và `ans` = số các `x` hợp lệ nhỏ hơn `n` mà khác `n` ở một bit cao hơn `k`.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time |
|---|---|---|
| 1 | Brute force (mục 5) | `O(n)` |
| 2 | Sinh trực tiếp các số hợp lệ (DFS) | `O(số đáp án)` |
| 3 | Digit DP có nhớ `(pos, prev, tight)` | `O(log n)` |
| 4 | Quét bit + Fibonacci | `O(log n)` |

### 7.1. Cách 2 — Sinh trực tiếp các số hợp lệ

**Giải thích.** Mọi số hợp lệ dương được tạo từ `1` bằng cách liên tục thêm bit vào bên phải: thêm `0` luôn được; thêm `1` chỉ khi bit cuối hiện tại là `0`. Dừng nhánh khi số vượt `n` (thêm bit chỉ làm số lớn hơn). Đếm số nút + 1 (cho số 0).

Cách này **nhanh hơn brute force** vì chỉ đi qua các số hợp lệ — nhưng vẫn tỉ lệ với đáp án (hàng triệu khi `n = 10^9`).

In [12]:
def find_integers_generate(n: int) -> int:
    count = 1                       # số 0
    stack = [1]
    while stack:
        x = stack.pop()
        if x > n:
            continue
        count += 1
        stack.append(x << 1)               # thêm bit 0
        if x & 1 == 0:
            stack.append((x << 1) | 1)     # thêm bit 1 (bit cuối đang là 0)
    return count

print([find_integers_generate(n) for n in range(1, 11)])
assert all(find_integers_generate(n) == find_integers_brute(n) for n in range(1, 300))

[2, 3, 3, 4, 5, 5, 5, 6, 7, 8]


**Đọc code:** `x << 1` = `2x` (thêm bit 0); `(x << 1) | 1` = `2x + 1` (thêm bit 1). `x & 1 == 0` nghĩa là bit cuối là 0. Mỗi số hợp lệ dương được sinh **đúng một lần** (cha của nó là nó bỏ bit cuối).

**Code nộp LeetCode (Cách 2):** có thể chậm với `n` lớn nhất (~2 triệu nút) nhưng thường vẫn qua.

In [13]:
class SolutionGenerate:
    def findIntegers(self, n: int) -> int:
        count, stack = 1, [1]
        while stack:
            x = stack.pop()
            if x <= n:
                count += 1
                stack.append(x << 1)
                if not x & 1:
                    stack.append(x << 1 | 1)
        return count

assert SolutionGenerate().findIntegers(5) == 5
t = time.perf_counter()
print("n=10^8:", SolutionGenerate().findIntegers(10**8), f"{time.perf_counter() - t:.2f}s")

n=10^8: 514229 0.05s


**Độ phức tạp Cách 2:** Time `O(A)` với `A` = đáp án (≈ `F(log₂ n)` ≈ `n^0.694`); Space `O(log n)` cho stack (DFS lặp; stack chứa tối đa vài phần tử mỗi tầng).

### 7.2. Cách 3 — Digit DP có nhớ

**Giải thích.** Coi `n` là chuỗi bit `bits` (từ cao xuống). Xây số `x` từng bit một, nhớ:

- `pos`: đang ở bit thứ mấy.
- `prev`: bit vừa đặt là 0 hay 1 (để cấm `11`).
- `tight`: các bit đã đặt có **bằng đúng** phần đầu của `n` không. Nếu `tight`, bit hiện tại không được vượt `bits[pos]`; nếu không `tight`, được đặt tự do.

```text
dfs(pos, prev, tight):
    nếu pos == len(bits): return 1            (đã xây xong một số hợp lệ)
    limit = bits[pos] nếu tight, ngược lại 1
    tổng = 0
    với d trong 0..limit:
        nếu d == 1 và prev == 1: bỏ qua
        tổng += dfs(pos+1, d, tight và d == limit)
    return tổng
Đáp án = dfs(0, 0, True)
```

In [14]:
def find_integers_digit_dp(n: int) -> int:
    bits = list(map(int, bin(n)[2:]))
    L = len(bits)

    @lru_cache(maxsize=None)
    def dfs(pos, prev, tight):
        if pos == L:
            return 1
        limit = bits[pos] if tight else 1
        total = 0
        for d in range(limit + 1):
            if d == 1 and prev == 1:
                continue
            total += dfs(pos + 1, d, tight and d == limit)
        return total

    return dfs(0, 0, True)

print([find_integers_digit_dp(n) for n in range(1, 11)])
assert all(find_integers_digit_dp(n) == find_integers_brute(n) for n in range(1, 500))

[2, 3, 3, 4, 5, 5, 5, 6, 7, 8]


**Đọc code:**

- `bits = list(map(int, bin(n)[2:]))`: chuỗi bit của `n`, bit cao trước.
- `tight and d == limit`: chỉ còn `tight` nếu vừa đặt đúng bằng bit của `n`.
- Số 0 cũng được đếm: đó là nhánh đặt toàn bit 0 (số có "số 0 ở đầu" chính là số ngắn hơn) — đúng ý, vì mọi số `< 2^L` đều viết được bằng `L` bit.
- Số trạng thái: `L × 2 × 2` ≤ 120 → cực nhanh.

**Code nộp LeetCode (Cách 3):**

In [15]:
class SolutionDigitDP:
    def findIntegers(self, n: int) -> int:
        bits = [int(c) for c in bin(n)[2:]]
        L = len(bits)

        @lru_cache(maxsize=None)
        def dfs(pos: int, prev: int, tight: bool) -> int:
            if pos == L:
                return 1
            limit = bits[pos] if tight else 1
            res = 0
            for d in range(limit + 1):
                if not (d and prev):
                    res += dfs(pos + 1, d, tight and d == limit)
            return res

        return dfs(0, 0, True)

assert SolutionDigitDP().findIntegers(5) == 5
assert SolutionDigitDP().findIntegers(10**9) == find_integers_digit_dp(10**9)
print("SolutionDigitDP OK; n = 10^9 ->", SolutionDigitDP().findIntegers(10**9))

SolutionDigitDP OK; n = 10^9 -> 2178309


**Độ phức tạp Cách 3:** `O(L · 2 · 2 · 2) = O(log n)` thời gian và bộ nhớ.

### 7.3. Cách 4 — Quét bit + Fibonacci

**Giải thích.** Đây là Digit DP được "giải tay": khi không còn `tight`, `dfs(pos, prev, False)` chỉ phụ thuộc số bit còn lại `k = L - pos` và `prev`; với `prev = 0` nó bằng `F(k)`. Nên ta chỉ cần đi dọc đường `tight` (bám theo `n`) và mỗi khi `n` có bit 1, cộng `F(k)` cho nhánh "đặt 0" (lúc đó `prev` mới = 0).

In [16]:
def find_integers_fib(n: int) -> int:
    f = [1, 2]
    for _ in range(2, 32):
        f.append(f[-1] + f[-2])
    ans, prev = 0, 0
    for k in range(31, -1, -1):           # bit 31 xuống bit 0
        if (n >> k) & 1:
            ans += f[k]                   # đặt 0 tại k, k bit thấp tự do
            if prev == 1:                 # bám theo n sẽ tạo "11" -> dừng
                return ans
            prev = 1
        else:
            prev = 0
    return ans + 1                        # chính n hợp lệ

print([find_integers_fib(n) for n in range(1, 11)])
assert all(find_integers_fib(n) == find_integers_brute(n) for n in range(1, 2000))

[2, 3, 3, 4, 5, 5, 5, 6, 7, 8]


**Đọc code:**

- `f[k]` = `F(k)`: số chuỗi `k` bit không có `11`. Cần tới `k = 31` vì `n <= 10^9 < 2^30`; lấy dư một chút cho an toàn.
- `ans += f[k]`: các số giống `n` ở mọi bit cao hơn `k`, có bit `k` = 0 (còn `n` có 1), và `k` bit thấp tùy ý (hợp lệ). Phần đầu (giống `n`) chưa có `11` theo invariant, và bit `k` = 0 không tạo `11` với bit trên.
- `prev == 1` khi gặp bit 1: `n` có `11` → mọi số tiếp tục bám theo `n` đều chứa `11` → dừng, **không** `+1`.
- Hết vòng: `n` không có `11` → `+1`.

**Code nộp LeetCode (Cách 4):**

In [17]:
class SolutionFib:
    def findIntegers(self, n: int) -> int:
        f = [1, 2] + [0] * 30
        for i in range(2, 32):
            f[i] = f[i - 1] + f[i - 2]
        ans, prev = 0, 0
        for k in range(n.bit_length() - 1, -1, -1):
            if n >> k & 1:
                ans += f[k]
                if prev:
                    return ans
                prev = 1
            else:
                prev = 0
        return ans + 1

assert SolutionFib().findIntegers(5) == 5
assert SolutionFib().findIntegers(1) == 2
assert SolutionFib().findIntegers(2) == 3
print("SolutionFib OK")

SolutionFib OK


Lưu ý `n >> k & 1`: toán tử `>>` có độ ưu tiên **cao hơn** `&` nên được hiểu là `(n >> k) & 1`. Viết ngoặc rõ ràng vẫn an toàn hơn.

**Độ phức tạp Cách 4:** Time `O(log n)` (≤ 32 vòng), Space `O(1)` (mảng 32 phần tử cố định).

## 8. TRACE từng bước

Trace **Cách 4** với `n = 21 = 10101₂` và `n = 13 = 1101₂`.

**`n = 21`:**

| k | bit k của n | hành động | ans | prev |
|---|---|---|---|---|
| 4 | 1 | `ans += F(4) = 8` | 8 | 1 |
| 3 | 0 | — | 8 | 0 |
| 2 | 1 | `ans += F(2) = 3` | 11 | 1 |
| 1 | 0 | — | 11 | 0 |
| 0 | 1 | `ans += F(0) = 1` | 12 | 1 |
| hết | | `n` hợp lệ: `+1` | **13** | |

**`n = 13`:**

| k | bit | hành động | ans | prev |
|---|---|---|---|---|
| 3 | 1 | `ans += F(3) = 5` | 5 | 1 |
| 2 | 1 | `ans += F(2) = 3`; prev = 1 → **dừng** | **8** | |

In [18]:
def trace_fib(n):
    f = F_list(32)
    ans, prev = 0, 0
    print(f"n = {n} = {n:b}")
    print(f"{'k':>3} {'bit':>4} {'hành động':<36} {'ans':>4} {'prev':>5}")
    for k in range(n.bit_length() - 1, -1, -1):
        b = (n >> k) & 1
        if b:
            ans += f[k]
            if prev:
                print(f"{k:>3} {b:>4} {'ans += F(%d) = %d; prev=1 -> DỪNG' % (k, f[k]):<36} {ans:>4}")
                return ans
            prev = 1
            print(f"{k:>3} {b:>4} {'ans += F(%d) = %d' % (k, f[k]):<36} {ans:>4} {prev:>5}")
        else:
            prev = 0
            print(f"{k:>3} {b:>4} {'-':<36} {ans:>4} {prev:>5}")
    print(f"hết bit: n hợp lệ -> ans + 1 = {ans + 1}")
    return ans + 1

assert trace_fib(21) == find_integers_brute(21) == 13
print()
assert trace_fib(13) == find_integers_brute(13) == 8

n = 21 = 10101
  k  bit hành động                             ans  prev
  4    1 ans += F(4) = 8                         8     1
  3    0 -                                       8     0
  2    1 ans += F(2) = 3                        11     1
  1    0 -                                      11     0
  0    1 ans += F(0) = 1                        12     1
hết bit: n hợp lệ -> ans + 1 = 13

n = 13 = 1101
  k  bit hành động                             ans  prev
  3    1 ans += F(3) = 5                         5     1
  2    1 ans += F(2) = 3; prev=1 -> DỪNG         8


Trace **Cách 3** (Digit DP) với `n = 5 = 101₂`: in cây lời gọi.

In [19]:
def trace_digit_dp(n):
    bits = list(map(int, bin(n)[2:]))
    L = len(bits)
    def dfs(pos, prev, tight, built, depth):
        pad = "  " * depth
        if pos == L:
            print(f"{pad}-> số {built} (= {int(built, 2)}) hợp lệ, trả 1")
            return 1
        limit = bits[pos] if tight else 1
        total = 0
        for d in range(limit + 1):
            if d == 1 and prev == 1:
                print(f"{pad}pos={pos}: thử d=1 nhưng prev=1 -> bỏ")
                continue
            print(f"{pad}pos={pos}, prev={prev}, tight={tight}: đặt bit {d}")
            total += dfs(pos + 1, d, tight and d == limit, built + str(d), depth + 1)
        return total
    return dfs(0, 0, True, "", 0)

assert trace_digit_dp(5) == 5

pos=0, prev=0, tight=True: đặt bit 0
  pos=1, prev=0, tight=False: đặt bit 0
    pos=2, prev=0, tight=False: đặt bit 0
      -> số 000 (= 0) hợp lệ, trả 1
    pos=2, prev=0, tight=False: đặt bit 1
      -> số 001 (= 1) hợp lệ, trả 1
  pos=1, prev=0, tight=False: đặt bit 1
    pos=2, prev=1, tight=False: đặt bit 0
      -> số 010 (= 2) hợp lệ, trả 1
    pos=2: thử d=1 nhưng prev=1 -> bỏ
pos=0, prev=0, tight=True: đặt bit 1
  pos=1, prev=1, tight=True: đặt bit 0
    pos=2, prev=0, tight=True: đặt bit 0
      -> số 100 (= 4) hợp lệ, trả 1
    pos=2, prev=0, tight=True: đặt bit 1
      -> số 101 (= 5) hợp lệ, trả 1


Trace **Cách 2** (sinh trực tiếp) với `n = 10`: in cây sinh số. Mỗi nút là một số hợp lệ; con trái thêm bit `0`, con phải thêm bit `1` (chỉ khi bit cuối là `0`). Nhánh vượt `n` bị cắt.

In [20]:
def trace_generate(n):
    count = 1
    print("0 (đếm sẵn)")
    def visit(x, depth):
        nonlocal count
        if x > n:
            print("    " * depth + f"{x:b} = {x} > {n}: cắt")
            return
        count += 1
        print("    " * depth + f"{x:b} = {x}  (đếm: {count})")
        visit(x << 1, depth + 1)
        if x & 1 == 0:
            visit((x << 1) | 1, depth + 1)
    visit(1, 0)
    return count

assert trace_generate(10) == 8

0 (đếm sẵn)
1 = 1  (đếm: 2)
    10 = 2  (đếm: 3)
        100 = 4  (đếm: 4)
            1000 = 8  (đếm: 5)
                10000 = 16 > 10: cắt
                10001 = 17 > 10: cắt
            1001 = 9  (đếm: 6)
                10010 = 18 > 10: cắt
        101 = 5  (đếm: 7)
            1010 = 10  (đếm: 8)
                10100 = 20 > 10: cắt
                10101 = 21 > 10: cắt


## 9. Code nộp LeetCode

Chọn Cách 4: `O(log n)`, không đệ quy, ngắn gọn.

In [21]:
class Solution:
    def findIntegers(self, n: int) -> int:
        # f[k] = số chuỗi k bit không có hai bit 1 liền nhau (Fibonacci 1, 2, 3, 5, 8, ...)
        f = [1, 2]
        for _ in range(2, 32):
            f.append(f[-1] + f[-2])
        ans, prev = 0, 0
        for k in range(n.bit_length() - 1, -1, -1):
            if (n >> k) & 1:
                ans += f[k]          # đặt bit k = 0, k bit thấp tự do
                if prev:             # n có "11": không thể tiếp tục bám theo n
                    return ans
                prev = 1
            else:
                prev = 0
        return ans + 1               # tính cả chính n

sol = Solution()
assert sol.findIntegers(5) == 5
assert sol.findIntegers(1) == 2
assert sol.findIntegers(2) == 3
print("n = 10^9 ->", sol.findIntegers(10**9))

n = 10^9 -> 2178309


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (Fibonacci).** `F(0) = 1`, `F(1) = 2`, `F(k) = F(k-1) + F(k-2)` đếm đúng số chuỗi `k` bit không có `11` — chứng minh ở mục 4.1 (phân loại theo bit đầu).

**Bổ đề 2 (phân hoạch).** Gọi `n` có `L` bit. Mỗi `x` trong `[0, n)` (viết đủ `L` bit) có **duy nhất** một vị trí `k` cao nhất mà `x` và `n` khác nhau; tại đó `n` có bit 1, `x` có bit 0 (vì `x < n`). Vậy tập `[0, n)` được chia thành các nhóm `G_k` theo vị trí `k` (chỉ với `k` là bit 1 của `n`), cộng thêm `{n}`.

**Đếm từng nhóm.** `x ∈ G_k` gồm: phần đầu = các bit của `n` cao hơn `k`, bit `k` = 0, phần thấp `k` bit tùy ý.
- Nếu phần đầu của `n` chứa `11` → mọi `x` trong nhóm đều không hợp lệ (đếm 0).
- Ngược lại, bit `k` = 0 cắt đứt, nên `x` hợp lệ ⇔ phần thấp hợp lệ → đúng `F(k)` số.

Thuật toán duyệt `k` giảm dần và dừng ở bit 1 đầu tiên làm phần đầu chứa `11` (tức hai bit 1 liền nhau trong `n`): các nhóm `G_k` đã cộng là đúng những nhóm có phần đầu sạch; các nhóm sau đó (và `n`) đều có phần đầu chứa `11` nên đóng góp 0. Nếu không dừng, `n` hợp lệ và được cộng 1. ∎

**Cách 3** đúng vì nó liệt kê có hệ thống mọi chuỗi `L` bit `<= n` không có `11` (trạng thái `(pos, prev, tight)` đủ để quyết định các bit còn lại). **Cách 2** đúng vì mỗi số hợp lệ dương có duy nhất một "cha" (bỏ bit cuối) cũng hợp lệ và nhỏ hơn, nên DFS từ `1` đi qua đúng một lần mọi số hợp lệ `<= n`.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n)` | `O(1)` | quá chậm với `10^9` |
| 2. Sinh số hợp lệ | `O(A)`, `A ≈ n^0.694` | `O(log n)` | ~2 triệu bước với `10^9` |
| 3. Digit DP | `O(log n)` | `O(log n)` | khuôn mẫu tổng quát |
| 4. Fibonacci | `O(log n)` | `O(1)` | **nộp bài** |

(`0.694 ≈ log₂ φ`, với `φ ≈ 1.618` là tỉ lệ vàng — vì `F(k) ≈ φ^k`.)

In [22]:
import random
random.seed(0)
qs = [random.randint(1, 10**9) for _ in range(20000)]
for name, f in [("digit_dp", lambda n: SolutionDigitDP().findIntegers(n)), ("fib", Solution().findIntegers)]:
    t = time.perf_counter()
    for q in qs:
        f(q)
    print(f"{name:>9}: 20000 truy vấn n ngẫu nhiên tới 10^9 trong {time.perf_counter() - t:.3f}s")

 digit_dp: 20000 truy vấn n ngẫu nhiên tới 10^9 trong 0.477s
      fib: 20000 truy vấn n ngẫu nhiên tới 10^9 trong 0.027s


## 12. Bộ test

In [23]:
impls = {
    "generate": find_integers_generate,
    "digit_dp": find_integers_digit_dp,
    "fib": find_integers_fib,
    "SolutionGenerate": lambda n: SolutionGenerate().findIntegers(n),
    "SolutionDigitDP": lambda n: SolutionDigitDP().findIntegers(n),
    "SolutionFib": lambda n: SolutionFib().findIntegers(n),
    "Solution": lambda n: Solution().findIntegers(n),
}

# Ví dụ đề
for name, f in impls.items():
    assert f(5) == 5 and f(1) == 2 and f(2) == 3, name

# Edge case: lũy thừa của 2, 2^k - 1 (toàn bit 1), dạng 1010...10, n lớn nhất
edge = [1, 2, 3, 4, 7, 8, 15, 16, 31, 32, 0b101010, 0b1010101, 2**20, 2**20 - 1]
for n in edge:
    b = find_integers_brute(n)
    assert all(f(n) == b for f in impls.values()), n
print("Ví dụ đề + edge case OK")

# Mọi n trong 1..3000
running = 1                                # đếm cho x = 0
for n in range(1, 3001):
    running += (n & (n >> 1)) == 0
    for name, f in impls.items():
        assert f(n) == running, (name, n)
print("Khớp brute force với mọi n trong 1..3000")

# Ngẫu nhiên có seed, n tới 2*10^5 (đối chiếu brute force)
random.seed(600)
for _ in range(30):
    n = random.randint(3001, 200000)
    b = find_integers_brute(n)
    assert all(f(n) == b for f in impls.values()), n
print("30 test ngẫu nhiên khớp brute force")

# n lớn: các cách O(log n) khớp nhau (và khớp generate ở một vài giá trị)
for _ in range(2000):
    n = random.randint(1, 10**9)
    assert find_integers_fib(n) == find_integers_digit_dp(n) == Solution().findIntegers(n)
for n in [10**7, 123456789]:
    assert find_integers_generate(n) == Solution().findIntegers(n)
print("n lớn: khớp")

Ví dụ đề + edge case OK


Khớp brute force với mọi n trong 1..3000
30 test ngẫu nhiên khớp brute force


n lớn: khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên dừng khi `n` có hai bit 1 liền nhau

In [24]:
def wrong_no_stop(n):
    f = F_list(32)
    ans, prev = 0, 0
    for k in range(n.bit_length() - 1, -1, -1):
        if (n >> k) & 1:
            ans += f[k]
            prev = 1                 # QUÊN kiểm tra prev trước đó
        else:
            prev = 0
    return ans + 1

print("n=3 sai:", wrong_no_stop(3), "| đúng:", find_integers_brute(3))
print("n=13 sai:", wrong_no_stop(13), "| đúng:", find_integers_brute(13))

n=3 sai: 4 | đúng: 3
n=13 sai: 10 | đúng: 8


**Vì sao sai:** khi `n` có `11`, các số tiếp tục "bám theo `n`" đều chứa `11` nhưng vẫn bị cộng (và cả `n` cũng bị `+1`).

**Code đúng:** gặp bit 1 mà `prev == 1` thì `return ans` ngay (không `+1`).

### Lỗi 2 — Dùng Fibonacci chuẩn `1, 1, 2, 3, ...` (lệch chỉ số)

In [25]:
def wrong_fib_index(n):
    f = [1, 1]                                 # SAI: F(1) phải là 2 (chuỗi "0" và "1")
    for _ in range(2, 33):
        f.append(f[-1] + f[-2])
    ans, prev = 0, 0
    for k in range(n.bit_length() - 1, -1, -1):
        if (n >> k) & 1:
            ans += f[k]
            if prev:
                return ans
            prev = 1
        else:
            prev = 0
    return ans + 1

print("n=5 sai:", wrong_fib_index(5), "| đúng:", find_integers_brute(5))

n=5 sai: 4 | đúng: 5


**Vì sao sai:** `f[k]` phải đếm chuỗi **`k` bit** không có `11`: `F(1) = 2` (`0`, `1`), `F(2) = 3`. Dùng dãy chuẩn bị lệch một chỉ số.

**Code đúng:** `f = [1, 2]` rồi cộng dồn. Luôn kiểm tra `F(k)` bằng vét cạn cho `k` nhỏ (như mục 4).

### Lỗi 3 — Kiểm tra bằng chuỗi nhưng quên số 0, hoặc đếm sai khoảng

In [26]:
def wrong_range(n):
    return sum(1 for x in range(1, n) if '11' not in bin(x))   # SAI: bỏ 0 và bỏ n

print("n=5 sai:", wrong_range(5), "| đúng:", find_integers_brute(5))

n=5 sai: 3 | đúng: 5


**Vì sao sai:** đề đếm trong đoạn **đóng** `[0, n]` — gồm cả 0 và `n`.

**Code đúng:** `range(n + 1)`.

### Lỗi 4 — Digit DP quên cập nhật `tight` đúng cách

In [27]:
def wrong_tight(n):
    bits = list(map(int, bin(n)[2:]))
    @lru_cache(maxsize=None)
    def dfs(pos, prev, tight):
        if pos == len(bits):
            return 1
        limit = bits[pos] if tight else 1
        return sum(dfs(pos + 1, d, tight) for d in range(limit + 1) if not (d and prev))  # SAI: giữ nguyên tight
    return dfs(0, 0, True)

print("n=5 sai:", wrong_tight(5), "| đúng:", find_integers_brute(5))

n=5 sai: 4 | đúng: 5


**Vì sao sai:** khi đặt bit **nhỏ hơn** bit của `n`, số đã chắc chắn nhỏ hơn `n` → các bit sau được tự do. Giữ `tight = True` làm các bit sau vẫn bị chặn bởi `n` → đếm thiếu.

**Code đúng:** `tight and d == limit`.

## 14. Mẹo & nhận dạng pattern

- **"Đếm số trong `[0, n]` (hoặc `[lo, hi]`) có tính chất về chữ số/bit"** → **Digit DP** với trạng thái `(vị trí, thông tin cần nhớ, tight)`. `[lo, hi]` = `count(hi) - count(lo - 1)`.
- **"Không có hai phần tử kề nhau cùng được chọn"** (bit 1 liền nhau, nhà liền kề, bậc thang 1–2 bước) → thường ra **Fibonacci**.
- Khi phần "tự do" có công thức đóng (như `F(k)`), Digit DP rút gọn thành **một vòng quét bit**.
- Kiểm tra nhanh "có hai bit 1 liền nhau": `x & (x >> 1)`.
- Luôn **vét cạn mọi `n` nhỏ** (vài nghìn) để bắt lỗi lệch chỉ số.

**Pattern mở rộng — đếm trên đoạn `[lo, hi]`.** Gần như mọi bài Digit DP hỏi trên đoạn đều quy về hai lần gọi hàm đếm tiền tố: `count(hi) - count(lo - 1)`. Cẩn thận khi `lo = 0` (khi đó `count(-1) = 0`).

In [28]:
def count_range(lo, hi):
    def upto(x):
        return 0 if x < 0 else Solution().findIntegers(x) if x > 0 else 1
    return upto(hi) - upto(lo - 1)

random.seed(11)
for _ in range(300):
    lo = random.randint(0, 3000)
    hi = random.randint(lo, 6000)
    assert count_range(lo, hi) == sum(1 for x in range(lo, hi + 1) if x & (x >> 1) == 0)
print("count_range khớp brute force; [10^8, 10^9] có", count_range(10**8, 10**9), "số")

count_range khớp brute force; [10^8, 10^9] có 1664080 số


## 15. Biến thể

1. **Đếm trong `[lo, hi]`:** `f(hi) - f(lo - 1)`.
2. **Không có `k` bit 1 liên tiếp:** trạng thái `prev` thành "số bit 1 liên tiếp ở cuối" (0..k-1) — Bài tập 5.
3. **Không có hai bit 0 liền nhau** (cho số có bit đầu là 1): đối xứng, cẩn thận số 0 đầu.
4. **Số thứ `k` không có `11`** (biểu diễn Zeckendorf): dùng `F` để quyết định từng bit từ cao xuống.
5. **Numbers At Most N Given Digit Set (LeetCode 902), Count Special Integers (LeetCode 2376):** Digit DP thập phân.

In [29]:
# Biến thể 4: số thứ k (0-indexed) trong dãy các số không có "11", tăng dần
def kth_no_consecutive(k):
    f = F_list(40)
    # tìm số bit L nhỏ nhất sao cho F(L) > k
    L = 0
    while f[L] <= k:
        L += 1
    x, prev = 0, 0
    for pos in range(L - 1, -1, -1):
        # thử đặt bit pos = 0: có f[pos] số với phần còn lại tự do
        if k < f[pos] or prev == 1:
            prev = 0
        else:
            k -= f[pos]
            x |= 1 << pos
            prev = 1
    return x

seq = [x for x in range(200) if x & (x >> 1) == 0]
assert [kth_no_consecutive(k) for k in range(len(seq))] == seq
print([kth_no_consecutive(k) for k in range(15)])

[0, 1, 2, 4, 5, 8, 9, 10, 16, 17, 18, 20, 21, 32, 33]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | kiểm tra `x & (x>>1)` cho mọi `x` | `O(n)` | `O(1)` | kiểm chứng `n` nhỏ |
| 2. Sinh số hợp lệ | DFS từ 1, thêm bit 0/1 | `O(A)` | `O(log n)` | khi cần **liệt kê** các số |
| 3. Digit DP | `dfs(pos, prev, tight)` + memo | `O(log n)` | `O(log n)` | khuôn mẫu cho mọi bài đếm theo chữ số |
| 4. Fibonacci | quét bit, mỗi bit 1 cộng `F(k)` | `O(log n)` | `O(1)` | **nộp bài** |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `Solution().findIntegers(8)` và `Solution().findIntegers(7)`?

**Bài 2 (tính tay).** Tính tay theo Cách 4 cho `n = 42 = 101010₂` (lập bảng như mục 8).

**Bài 3 (điền TODO).** Hoàn thiện `fib_counts(K)` trả về list `F(0..K)` và tự kiểm tra bằng vét cạn với `K <= 10`.

**Bài 4 (tự viết lại).** Viết lại Digit DP **dạng lặp** (không đệ quy), duyệt bit từ cao xuống, giữ số cách ở trạng thái `(prev, tight)`.

**Bài 5 (mini challenge).** Viết `count_no_k_consecutive(n, k)`: đếm `x ∈ [0, n]` không có `k` bit 1 liên tiếp (với `k = 2` chính là bài gốc).

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def fib_counts(K):
    # TODO: f = [1, 2], cộng dồn tới K
    pass

# Bài 4 — TODO
def find_integers_iterative(n):
    # TODO: ways[(prev, tight)]; bắt đầu {(0, True): 1}; mỗi bit cập nhật sang dict mới
    pass

# Bài 5 — TODO
def count_no_k_consecutive(n, k):
    # TODO: Digit DP với trạng thái run = số bit 1 liên tiếp ở cuối (0..k-1)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: liệt kê các số `<= 8`.
- Mức 2: `8 = 1000₂` → `F(3) + 1`.
- Mức 3: `7 = 111₂` → `F(2) + F(1)` rồi dừng.

**Bài 2.**
- Mức 1: các bit 1 ở vị trí 5, 3, 1.
- Mức 2: không có hai bit 1 liền nhau → không dừng sớm.
- Mức 3: `F(5) + F(3) + F(1) + 1`.

**Bài 3.**
- Mức 1: `f[k] = f[k-1] + f[k-2]`.
- Mức 2: vét cạn `range(2**k)` với `x & (x >> 1) == 0`.
- Mức 3: `assert` cho từng `k`.

**Bài 4.**
- Mức 1: trạng thái nhỏ: `prev ∈ {0,1}`, `tight ∈ {True, False}`.
- Mức 2: với mỗi trạng thái và mỗi `d` hợp lệ, cộng số cách vào trạng thái mới.
- Mức 3: đáp án = tổng mọi trạng thái sau bit cuối.

**Bài 5.**
- Mức 1: thay `prev` bằng `run`.
- Mức 2: đặt `d = 1` thì `run + 1` (phải `< k`); đặt `d = 0` thì `run = 0`.
- Mức 3: đối chiếu với vét cạn `'1' * k not in bin(x)`.

## 19. Đáp án tham khảo

**Bài 1:** `n = 8` → `F(3) + 1 = 6`; `n = 7` → `F(2) + F(1) = 3 + 2 = 5` (dừng ở bit 1).

**Bài 2:** `F(5) + F(3) + F(1) + 1 = 13 + 5 + 2 + 1 = 21`.

In [31]:
assert Solution().findIntegers(8) == 6 == find_integers_brute(8)
assert Solution().findIntegers(7) == 5 == find_integers_brute(7)
assert Solution().findIntegers(42) == 21 == find_integers_brute(42)

# Bài 3
def fib_counts(K):
    f = [1, 2]
    while len(f) <= K:
        f.append(f[-1] + f[-2])
    return f[:K + 1]

for k in range(11):
    assert fib_counts(10)[k] == sum(1 for x in range(2 ** k) if x & (x >> 1) == 0)

# Bài 4
def find_integers_iterative(n):
    ways = {(0, True): 1}
    for b in map(int, bin(n)[2:]):
        new = {}
        for (prev, tight), c in ways.items():
            limit = b if tight else 1
            for d in range(limit + 1):
                if d and prev:
                    continue
                key = (d, tight and d == limit)
                new[key] = new.get(key, 0) + c
        ways = new
    return sum(ways.values())

assert all(find_integers_iterative(n) == find_integers_brute(n) for n in range(1, 2000))
print("Bài 1–4 OK")

Bài 1–4 OK


In [32]:
# Bài 5
def count_no_k_consecutive(n, k):
    bits = list(map(int, bin(n)[2:]))
    L = len(bits)

    @lru_cache(maxsize=None)
    def dfs(pos, run, tight):
        if pos == L:
            return 1
        limit = bits[pos] if tight else 1
        total = 0
        for d in range(limit + 1):
            nr = run + 1 if d else 0
            if nr >= k:
                continue
            total += dfs(pos + 1, nr, tight and d == limit)
        return total

    return dfs(0, 0, True)

for k in (1, 2, 3, 4):
    for n in range(1, 600):
        brute = sum(1 for x in range(n + 1) if '1' * k not in bin(x)[2:])
        assert count_no_k_consecutive(n, k) == brute, (n, k)
assert all(count_no_k_consecutive(n, 2) == Solution().findIntegers(n) for n in range(1, 3000))
print("Bài 5 OK; n=10^9, k=3:", count_no_k_consecutive(10**9, 3))

Bài 5 OK; n=10^9, k=3: 98950096


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Digit DP (nhị phân):**

```python
bits = list(map(int, bin(n)[2:]))
@lru_cache(None)
def dfs(pos, state, tight):
    if pos == len(bits): return 1 if ok(state) else 0
    limit = bits[pos] if tight else 1
    return sum(dfs(pos + 1, nxt(state, d), tight and d == limit)
               for d in range(limit + 1) if allowed(state, d))
return dfs(0, start_state, True)
```

**Template quét bit + Fibonacci (bài này):**

```python
f = [1, 2]; [f.append(f[-1] + f[-2]) for _ in range(30)]
ans = prev = 0
for k in range(n.bit_length() - 1, -1, -1):
    if n >> k & 1:
        ans += f[k]
        if prev: return ans
        prev = 1
    else:
        prev = 0
return ans + 1
```

**Câu thần chú:** *"Gặp bit 1: đặt 0 thì phần dưới tự do (Fibonacci); hai bit 1 liền thì dừng; đi hết thì cộng chính n."*

**Checklist trước khi submit:**

- [ ] `f = [1, 2, 3, 5, ...]` (không phải `1, 1, 2, ...`).
- [ ] Mảng `f` đủ dài cho `n.bit_length()` bit.
- [ ] Dừng sớm (không `+1`) khi gặp hai bit 1 liền nhau.
- [ ] Đếm cả 0 và `n` (đoạn đóng `[0, n]`).
- [ ] Đã thử `n = 1 → 2`, `2 → 3`, `3 → 3`, `5 → 5`, `8 → 6`.
- [ ] Đối chiếu brute force cho mọi `n` vài nghìn đầu tiên.